# Evaluate Benchmarks


In [ ]:

import json
import os
import re
import sys
from pathlib import Path

import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.utils import get_column_letter
from tqdm import tqdm

BASE_DIR = Path.cwd()
SCORE_MODULE_DIR = BASE_DIR
if str(SCORE_MODULE_DIR) not in sys.path:
    sys.path.insert(0, str(SCORE_MODULE_DIR))

import score


In [ ]:

MODEL_NAME = os.environ.get('MODEL_NAME', 'model-placeholder')

BENCHMARK_DIR = BASE_DIR / 'Benchmarks'
RAW_RESULTS_DIR = BASE_DIR / 'Outputs' / 'Raw Results'
EVALUATED_RESULTS_DIR = BASE_DIR / 'Outputs' / 'Evaluated Results'

RAW_OUTPUT_PATH = RAW_RESULTS_DIR / f'{MODEL_NAME}_benchmark_outputs.xlsx'
GOLD_PATH = BENCHMARK_DIR / 'Reference Answer.xlsx'
EVALUATION_OUTPUT_PATH = EVALUATED_RESULTS_DIR / f'{MODEL_NAME}_evaluation_results.xlsx'

MODEL_OUTPUT_COLUMN = f'{MODEL_NAME}_output'
MODEL_STATUS_COLUMN = f'{MODEL_NAME}_status'
MODEL_ERROR_COLUMN = f'{MODEL_NAME}_error'
MODEL_ELAPSED_SECONDS_COLUMN = f'{MODEL_NAME}_elapsed_seconds'

MTBENCH_CONFIG = {
    'benchmark': 'MTbench',
    'sheet_name': 'MTbench_cases',
    'id_column': 'Case_ID',
}


MTBENCH_METRIC_GROUPS = {
    'categorical_field_accuracy': {'accuracy'},
    'multi_label_field_f1': {'multilabel_f1', 'combination_type_f1'},
    'temporal_structure_f1': {'duration_f1', 'frequency_f1', 'study_period_f1'},
    'bpm_mae': {'bpm_mae'},
}

EXTEND_BENCHMARKS = [
    {
        'benchmark': 'TGM',
        'display_name': 'Treatment Goal Matching',
        'sheet_name': 'TGM_cases',
        'id_column': 'TGM_ID',
        'gold_column': 'Correct_Option',
        'mode': 'single',
        'option_letters': 'ABCD',
        'result_sheet': 'TGM_results',
    },
    {
        'benchmark': 'SM',
        'display_name': 'Song Matching',
        'sheet_name': 'SM_cases',
        'id_column': 'SM_ID',
        'gold_column': 'Correct_Options',
        'mode': 'multi',
        'option_letters': 'ABCDEFGH',
        'result_sheet': 'SM_results',
    },
    {
        'benchmark': 'PC',
        'display_name': 'Prescription Completion',
        'sheet_name': 'PC_cases',
        'id_column': 'PC_ID',
        'gold_column': 'Correct_Option',
        'mode': 'single',
        'option_letters': 'ABCD',
        'result_sheet': 'PC_results',
    },
    {
        'benchmark': 'CCM',
        'display_name': 'Clinical Context Matching',
        'sheet_name': 'CCM_cases',
        'id_column': 'CCM_ID',
        'gold_column': 'Correct_Option',
        'mode': 'single',
        'option_letters': 'ABCD',
        'result_sheet': 'CCM_results',
    },
]


In [ ]:

def require_file(path):
    if not path.exists():
        raise FileNotFoundError(f'Required file not found: {path}')



def require_columns(df, columns, label):
    missing = [column for column in columns if column not in df.columns]
    if missing:
        raise ValueError(f'{label} missing required columns: {missing}')



def clean_text(value):
    if pd.isna(value):
        return ''
    return str(value).strip()



def find_json_key(obj, target_name):
    if not isinstance(obj, dict):
        return None
    target_lower = target_name.lower()
    for key in obj:
        if str(key).lower() == target_lower:
            return key
    return None


def stringify_payload(value):
    if isinstance(value, (dict, list)):
        return json.dumps(value, ensure_ascii=False)
    return clean_text(value)


def unwrap_answer_object(obj):
    if not isinstance(obj, dict):
        return obj, '', False
    answer_key = find_json_key(obj, 'Answer')
    reasoning_key = find_json_key(obj, 'Reasoning')
    reasoning = stringify_payload(obj.get(reasoning_key, '')) if reasoning_key is not None else ''
    if answer_key is None:
        return obj, reasoning, False
    return obj.get(answer_key, ''), reasoning, True


def extract_answer_payload(raw_output, require_answer_field=False):
    text = clean_text(raw_output)
    if not text:
        return '', '', 'Missing model output'
    try:
        obj = json.loads(text)
    except Exception:
        return text, '', ''
    answer, reasoning, has_answer_field = unwrap_answer_object(obj)
    if require_answer_field and isinstance(obj, dict) and not has_answer_field:
        return '', reasoning, 'JSON output missing Answer field'
    return answer, reasoning, ''



def mean_or_none(values):
    series = pd.to_numeric(pd.Series(values), errors='coerce').dropna()
    if series.empty:
        return None
    return round(float(series.mean()), 6)


def clean_elapsed_seconds(value):
    series = pd.to_numeric(pd.Series([value]), errors='coerce').dropna()
    if series.empty:
        return None
    return round(float(series.iloc[0]), 2)


def row_elapsed_seconds(row):
    if MODEL_ELAPSED_SECONDS_COLUMN not in row.index:
        return None
    return clean_elapsed_seconds(row.get(MODEL_ELAPSED_SECONDS_COLUMN))



def elapsed_summary_values(df):
    if MODEL_ELAPSED_SECONDS_COLUMN not in df.columns:
        return None, None
    series = pd.to_numeric(df[MODEL_ELAPSED_SECONDS_COLUMN], errors='coerce').dropna()
    if series.empty:
        return None, None
    return round(float(series.mean()), 2), round(float(series.sum()), 2)



def write_evaluation_workbook(sheet_frames, output_path):
    output_path.parent.mkdir(parents=True, exist_ok=True)
    with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
        for sheet_name, frame in sheet_frames.items():
            frame.to_excel(writer, sheet_name=sheet_name, index=False)

    workbook = load_workbook(output_path)
    header_fill = PatternFill(fill_type='solid', fgColor='D9EAF7')
    header_font = Font(bold=True)
    for worksheet in workbook.worksheets:
        worksheet.freeze_panes = 'A2'
        if worksheet.max_row >= 1 and worksheet.max_column >= 1:
            worksheet.auto_filter.ref = worksheet.dimensions
        for cell in worksheet[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center')
        for column_idx in range(1, worksheet.max_column + 1):
            letter = get_column_letter(column_idx)
            values = [worksheet.cell(row=row_idx, column=column_idx).value for row_idx in range(1, min(worksheet.max_row, 80) + 1)]
            width = min(max([len(str(value)) for value in values if value is not None] + [10]) + 2, 60)
            worksheet.column_dimensions[letter].width = width
    workbook.save(output_path)


In [ ]:

def evaluate_mtbench(raw_workbook_path, gold_path):
    case_df = pd.read_excel(raw_workbook_path, sheet_name=MTBENCH_CONFIG['sheet_name'])
    gold_df = pd.read_excel(gold_path, sheet_name='gold_card')
    require_columns(case_df, [MTBENCH_CONFIG['id_column'], MODEL_OUTPUT_COLUMN, MODEL_STATUS_COLUMN, MODEL_ERROR_COLUMN], MTBENCH_CONFIG['sheet_name'])
    require_columns(gold_df, ['Case_ID'] + [spec['gold_col'] for spec in score.FIELD_SPECS], 'gold_card')

    gold_by_case = {score.clean_value(row['Case_ID']): row for _, row in gold_df.iterrows()}
    detail_rows = []
    numeric_rows = []
    parse_fail_rows = 0

    iterator = tqdm(case_df.iterrows(), total=len(case_df), desc='MTbench evaluation', unit='row')
    for _, case_row in iterator:
        case_id = score.clean_value(case_row[MTBENCH_CONFIG['id_column']])
        if case_id not in gold_by_case:
            raise ValueError(f'{case_id} was not found in gold_card.')

        raw_output = case_row[MODEL_OUTPUT_COLUMN]
        run_status = clean_text(case_row[MODEL_STATUS_COLUMN])
        run_error = clean_text(case_row[MODEL_ERROR_COLUMN])
        elapsed_seconds = row_elapsed_seconds(case_row)
        model_reasoning = ''
        if not score.has_model_output(raw_output):
            model_obj = None
            parse_error = 'Missing model output'
        else:
            parsed_obj, parse_error = score.parse_json(score.clean_value(raw_output))
            if parse_error:
                model_obj = None
            else:
                model_obj, model_reasoning, _ = unwrap_answer_object(parsed_obj)

        if parse_error:
            parse_fail_rows += 1

        detail_row = {
            'Case_ID': case_id,
            'run_status': run_status,
            'parse_error': parse_error,
            'run_error': run_error,
            'elapsed_seconds': elapsed_seconds,
            'model_reasoning': model_reasoning,
        }
        numeric_row = {'Case_ID': case_id, 'run_status': run_status, 'parse_error': parse_error, 'elapsed_seconds': elapsed_seconds}
        gold_row = gold_by_case[case_id]
        for spec in score.FIELD_SPECS:
            if parse_error:
                field_result = score.failed_field_json(spec, gold_row, parse_error)
            else:
                field_result = score.score_field(spec, gold_row, model_obj)
            detail_row[spec['name']] = field_result
            numeric_row[spec['name']] = score.extract_score(field_result)

        detail_rows.append(detail_row)
        numeric_rows.append(numeric_row)

    detail_df = pd.DataFrame(detail_rows)
    numeric_df = pd.DataFrame(numeric_rows)
    field_mean_rows = []
    field_means = {}
    for spec in score.FIELD_SPECS:
        field_name = spec['name']
        field_mean = mean_or_none(numeric_df[field_name])
        field_means[field_name] = field_mean
        field_mean_rows.append({
            'field': field_name,
            'metric': spec['metric'],
            'field_mean': field_mean,
        })

    metric_summary = {}
    metric_rows = []
    for metric_family, metric_names in MTBENCH_METRIC_GROUPS.items():
        group_fields = [spec['name'] for spec in score.FIELD_SPECS if spec['metric'] in metric_names]
        group_score = mean_or_none([field_means[field] for field in group_fields])
        metric_summary[metric_family] = group_score
        metric_rows.append({
            'metric_family': metric_family,
            'score': group_score,
            'direction': 'lower_is_better' if metric_family == 'bpm_mae' else 'higher_is_better',
            'fields': '; '.join(group_fields),
        })

    field_mean_df = pd.DataFrame(field_mean_rows)
    metric_df = pd.DataFrame(metric_rows)
    primary_score_payload = {
        'categorical_field_accuracy': metric_summary['categorical_field_accuracy'],
        'multi_label_field_f1': metric_summary['multi_label_field_f1'],
        'temporal_structure_f1': metric_summary['temporal_structure_f1'],
        'bpm_mae': metric_summary['bpm_mae'],
    }
    avg_elapsed_seconds, total_elapsed_seconds = elapsed_summary_values(case_df)
    extra_metric_payload = {
        'field_means': {row['field']: row['field_mean'] for row in field_mean_rows},
        'parse_fail_rows': parse_fail_rows,
        'avg_elapsed_seconds': avg_elapsed_seconds,
        'total_elapsed_seconds': total_elapsed_seconds,
    }
    summary_row = {
        'benchmark': 'MTbench',
        'rows': len(case_df),
        'success_rows': int((case_df[MODEL_STATUS_COLUMN] == 'success').sum()),
        'fail_rows': int((case_df[MODEL_STATUS_COLUMN] != 'success').sum()),
        'primary_metric': 'metric_family_scores',
        'primary_score': json.dumps(primary_score_payload, ensure_ascii=False),
        'avg_elapsed_seconds': avg_elapsed_seconds,
        'total_elapsed_seconds': total_elapsed_seconds,
        'extra_metrics': json.dumps(extra_metric_payload, ensure_ascii=False),
    }
    return detail_df, numeric_df, field_mean_df, metric_df, summary_row


In [ ]:

def extract_options(value, option_letters):
    text = clean_text(value).upper()
    if not text:
        return []
    allowed = ''.join(dict.fromkeys(option_letters.upper()))
    pattern = rf'(?<![A-Z])[{re.escape(allowed)}](?![A-Z])'
    tokens = re.findall(pattern, text)
    if not tokens:
        compact = re.sub(r'[\s,;，、/|+&\-\.\(\)\[\]{}:：]+', '', text)
        if re.fullmatch(rf'[{re.escape(allowed)}]+', compact or ''):
            tokens = list(compact)
    return list(dict.fromkeys(tokens))



def option_parse_error(cfg):
    return f"No {cfg['option_letters'][0]}-{cfg['option_letters'][-1]} option found in model output"


def set_metrics(gold_options, model_options):
    gold_set = set(gold_options)
    model_set = set(model_options)
    intersection = gold_set & model_set
    precision = 0.0 if not model_set else len(intersection) / len(model_set)
    recall = 0.0 if not gold_set else len(intersection) / len(gold_set)
    f1 = 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)
    return round(precision, 6), round(recall, 6), round(f1, 6)


def evaluate_single_choice(raw_workbook_path, cfg):
    df = pd.read_excel(raw_workbook_path, sheet_name=cfg['sheet_name'])
    require_columns(df, [cfg['id_column'], cfg['gold_column'], MODEL_OUTPUT_COLUMN, MODEL_STATUS_COLUMN, MODEL_ERROR_COLUMN], cfg['sheet_name'])
    rows = []
    iterator = tqdm(df.iterrows(), total=len(df), desc=f"{cfg['benchmark']} evaluation", unit='row')
    for _, row in iterator:
        gold_options = extract_options(row[cfg['gold_column']], cfg['option_letters'])
        elapsed_seconds = row_elapsed_seconds(row)
        answer_payload, model_reasoning, answer_parse_error = extract_answer_payload(row[MODEL_OUTPUT_COLUMN], require_answer_field=True)
        model_options = extract_options(answer_payload, cfg['option_letters'])
        parse_error = answer_parse_error
        if not parse_error:
            if len(model_options) == 0:
                parse_error = option_parse_error(cfg)
            elif len(model_options) > 1:
                parse_error = f'Expected one option, found {model_options}'
        is_correct = len(gold_options) == 1 and len(model_options) == 1 and gold_options[0] == model_options[0]
        rows.append({
            cfg['id_column']: row[cfg['id_column']],
            'gold_answer': '; '.join(gold_options),
            'model_answer': '; '.join(model_options),
            'is_correct': int(is_correct),
            'model_answer_raw': stringify_payload(answer_payload),
            'model_reasoning': model_reasoning,
            'elapsed_seconds': elapsed_seconds,
            'model_raw': clean_text(row[MODEL_OUTPUT_COLUMN]),
            'run_status': clean_text(row[MODEL_STATUS_COLUMN]),
            'parse_error': parse_error,
            'run_error': clean_text(row[MODEL_ERROR_COLUMN]),
        })

    result_df = pd.DataFrame(rows)
    avg_elapsed_seconds, total_elapsed_seconds = elapsed_summary_values(df)
    summary_row = {
        'benchmark': cfg['benchmark'],
        'rows': len(result_df),
        'success_rows': int((df[MODEL_STATUS_COLUMN] == 'success').sum()),
        'fail_rows': int((df[MODEL_STATUS_COLUMN] != 'success').sum()),
        'primary_metric': 'accuracy',
        'primary_score': mean_or_none(result_df['is_correct']),
        'avg_elapsed_seconds': avg_elapsed_seconds,
        'total_elapsed_seconds': total_elapsed_seconds,
        'extra_metrics': json.dumps({
            'parse_fail_rows': int((result_df['parse_error'] != '').sum()),
            'avg_elapsed_seconds': avg_elapsed_seconds,
            'total_elapsed_seconds': total_elapsed_seconds,
        }, ensure_ascii=False),
    }
    return result_df, summary_row



def evaluate_multi_choice(raw_workbook_path, cfg):
    df = pd.read_excel(raw_workbook_path, sheet_name=cfg['sheet_name'])
    require_columns(df, [cfg['id_column'], cfg['gold_column'], MODEL_OUTPUT_COLUMN, MODEL_STATUS_COLUMN, MODEL_ERROR_COLUMN], cfg['sheet_name'])
    rows = []
    iterator = tqdm(df.iterrows(), total=len(df), desc=f"{cfg['benchmark']} evaluation", unit='row')
    for _, row in iterator:
        gold_options = extract_options(row[cfg['gold_column']], cfg['option_letters'])
        elapsed_seconds = row_elapsed_seconds(row)
        answer_payload, model_reasoning, answer_parse_error = extract_answer_payload(row[MODEL_OUTPUT_COLUMN], require_answer_field=True)
        model_options = extract_options(answer_payload, cfg['option_letters'])
        precision, recall, f1 = set_metrics(gold_options, model_options)
        exact_match = set(gold_options) == set(model_options)
        parse_error = answer_parse_error
        if not parse_error and not model_options:
            parse_error = option_parse_error(cfg)
        rows.append({
            cfg['id_column']: row[cfg['id_column']],
            'gold_answer': '; '.join(gold_options),
            'model_answer': '; '.join(model_options),
            'exact_match': int(exact_match),
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'model_answer_raw': stringify_payload(answer_payload),
            'model_reasoning': model_reasoning,
            'elapsed_seconds': elapsed_seconds,
            'model_raw': clean_text(row[MODEL_OUTPUT_COLUMN]),
            'run_status': clean_text(row[MODEL_STATUS_COLUMN]),
            'parse_error': parse_error,
            'run_error': clean_text(row[MODEL_ERROR_COLUMN]),
        })

    result_df = pd.DataFrame(rows)
    avg_elapsed_seconds, total_elapsed_seconds = elapsed_summary_values(df)
    summary_row = {
        'benchmark': cfg['benchmark'],
        'rows': len(result_df),
        'success_rows': int((df[MODEL_STATUS_COLUMN] == 'success').sum()),
        'fail_rows': int((df[MODEL_STATUS_COLUMN] != 'success').sum()),
        'primary_metric': 'exact_match_accuracy',
        'primary_score': mean_or_none(result_df['exact_match']),
        'avg_elapsed_seconds': avg_elapsed_seconds,
        'total_elapsed_seconds': total_elapsed_seconds,
        'extra_metrics': json.dumps({
            'option_precision_mean': mean_or_none(result_df['precision']),
            'option_recall_mean': mean_or_none(result_df['recall']),
            'option_f1_mean': mean_or_none(result_df['f1']),
            'parse_fail_rows': int((result_df['parse_error'] != '').sum()),
            'avg_elapsed_seconds': avg_elapsed_seconds,
            'total_elapsed_seconds': total_elapsed_seconds,
        }, ensure_ascii=False),
    }
    return result_df, summary_row



In [ ]:

def main():
    require_file(RAW_OUTPUT_PATH)
    require_file(GOLD_PATH)

    mtbench_detail_df, mtbench_numeric_df, mtbench_field_mean_df, mtbench_metric_df, mtbench_summary = evaluate_mtbench(RAW_OUTPUT_PATH, GOLD_PATH)
    summary_rows = [mtbench_summary]
    sheet_frames = {
        'summary': None,
        'MTbench_field_scores': mtbench_detail_df,
        'MTbench_numeric_scores': mtbench_numeric_df,
        'MTbench_field_means': mtbench_field_mean_df,
        'MTbench_metric_scores': mtbench_metric_df,
    }

    for cfg in EXTEND_BENCHMARKS:
        if cfg['mode'] == 'single':
            result_df, summary_row = evaluate_single_choice(RAW_OUTPUT_PATH, cfg)
        elif cfg['mode'] == 'multi':
            result_df, summary_row = evaluate_multi_choice(RAW_OUTPUT_PATH, cfg)
        else:
            raise ValueError(f"Unknown benchmark mode: {cfg['mode']}")
        sheet_frames[cfg['result_sheet']] = result_df
        summary_rows.append(summary_row)

    summary_df = pd.DataFrame(summary_rows)
    sheet_frames['summary'] = summary_df
    write_evaluation_workbook(sheet_frames, EVALUATION_OUTPUT_PATH)
    print(f'Evaluation results saved to: {EVALUATION_OUTPUT_PATH}')
    return summary_df



summary = main()
summary
